## Creates the **Epidemiological Timeline** for a specific region by using the available data on WNV Cases ##

In [1]:
import pandas as pd
import numpy as np

enc = 'utf-8'
dec = 'greek8'

pd.options.display.max_columns = None
pd.options.display.max_rows = 10

In [2]:
NUTS0 = 'GR'
NUTS2 = 'Attica'
NUTS2_EL = 'αττικης'

In [3]:
data = pd.read_csv(f'../../data/{NUTS0}_WNV_cases_2010-2022_processed.csv', encoding = enc)
data.head(5)

,onset of symptoms,year,month,day,nuts2,nuts3,cases
0,2010-07-06,2010,7,6,κεντρικης μακεδονιας,σερρων,1
1,2010-07-16,2010,7,16,κεντρικης μακεδονιας,κιλκις,1
2,2010-07-18,2010,7,18,κεντρικης μακεδονιας,πελλας,1
3,2010-07-19,2010,7,19,κεντρικης μακεδονιας,θεσσαλονικης,2
4,2010-07-20,2010,7,20,κεντρικης μακεδονιας,ημαθιας,1


In [4]:
data.shape

(1306, 7)

In [5]:
df = data[data['nuts2'].isin([NUTS2_EL])].copy()

In [6]:
df.reset_index(drop = True, inplace = True)

In [7]:
df.drop(columns = ['onset of symptoms', 'day'], inplace = True)

In [8]:
df

,year,month,nuts2,nuts3,cases
0,2011,7,αττικης,ανατολικης αττικης,1
1,2011,7,αττικης,ανατολικης αττικης,2
2,2011,7,αττικης,ανατολικης αττικης,2
3,2011,7,αττικης,ανατολικης αττικης,1
4,2011,7,αττικης,ανατολικης αττικης,1
...,...,...,...,...,...
239,2020,7,αττικης,ανατολικης αττικης,1
240,2020,8,αττικης,ανατολικης αττικης,1
241,2021,7,αττικης,ανατολικης αττικης,1
242,2021,8,αττικης,ανατολικης αττικης,1


In [9]:
df.cases.sum()

296

In [10]:
df_grouped = df.groupby(['year', 'month', 'nuts2', 'nuts3'], as_index = False)
df = df_grouped.sum()

In [11]:
df

,year,month,nuts2,nuts3,cases
0,2011,7,αττικης,ανατολικης αττικης,7
1,2011,8,αττικης,ανατολικης αττικης,8
2,2011,8,αττικης,βορειου τομεα αθηνων,1
3,2011,8,αττικης,δυτικης αττικης,1
4,2011,8,αττικης,πειραιως & νησων,1
...,...,...,...,...,...
62,2020,7,αττικης,ανατολικης αττικης,1
63,2020,8,αττικης,ανατολικης αττικης,1
64,2021,7,αττικης,ανατολικης αττικης,1
65,2021,8,αττικης,ανατολικης αττικης,1


In [12]:
df.cases.sum()

296

In [13]:
print(np.sort(df.cases.unique()))

[ 1  2  3  4  5  6  7  8  9 10 11 12 15 16 18 20 33]


In [14]:
wnv_lau1_list = df['nuts3'].drop_duplicates().sort_values().tolist()

with open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_2010-2022_NUTS3_Units_Processed.txt', 'w', encoding=enc) as f:
    for item in wnv_lau1_list:
        f.write("%s\n" % item)
  
print(f"Number of NUTS3 Units in {NUTS2} (from cases): {len(wnv_lau1_list)}")

Number of NUTS3 Units in Attica (from cases): 7


In [15]:
df['date_string'] = df['month'].astype(str) + '-' + df['year'].astype(str)

# Convert 'date_string' column to datetime format
df['dt_placement'] = pd.to_datetime(df['date_string'], format='%m-%Y').dt.to_period('M')
df.drop(columns=['date_string'], inplace = True)

In [16]:
df

,year,month,nuts2,nuts3,cases,dt_placement
0,2011,7,αττικης,ανατολικης αττικης,7,2011-07
1,2011,8,αττικης,ανατολικης αττικης,8,2011-08
2,2011,8,αττικης,βορειου τομεα αθηνων,1,2011-08
3,2011,8,αττικης,δυτικης αττικης,1,2011-08
4,2011,8,αττικης,πειραιως & νησων,1,2011-08
...,...,...,...,...,...,...
62,2020,7,αττικης,ανατολικης αττικης,1,2020-07
63,2020,8,αττικης,ανατολικης αττικης,1,2020-08
64,2021,7,αττικης,ανατολικης αττικης,1,2021-07
65,2021,8,αττικης,ανατολικης αττικης,1,2021-08


In [17]:
df.cases.sum()

296

In [18]:
case_months = df['month'].drop_duplicates().sort_values().tolist()
case_months

[5, 6, 7, 8, 9, 10]

In [19]:
case_years = df['year'].drop_duplicates().sort_values().tolist()
case_years

[2011, 2012, 2013, 2014, 2018, 2019, 2020, 2021]

In [20]:
with open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_2010-2022_NUTS3_Cases_Dates.txt', 'w', encoding=enc) as f:

    f.write("%s: " % 'months')
    for item in case_months:
        f.write("%s, " % item)

    f.write("\n")

    f.write("%s: " % 'years')
    for item in case_years:
        f.write("%s, " % item)

In [21]:
df.drop(columns=['month', 'year'], inplace = True)

In [22]:
## Creating DataFrame to hold negative examples

df_timeline = pd.DataFrame(columns = ['nuts2', 'nuts3', 'dt_placement', 'cases'])
timeline = []

for nuts3 in df['nuts3'].drop_duplicates().sort_values():
    for year in case_years:
        for month in case_months:
            timeline.append({'nuts2' : NUTS2_EL, 'nuts3' : nuts3, 'dt_placement' : f"{year:04}-{month:02}", 'cases' : 0})
            
df_timeline = pd.DataFrame(timeline)

In [23]:
df_timeline

,nuts2,nuts3,dt_placement,cases
0,αττικης,ανατολικης αττικης,2011-05,0
1,αττικης,ανατολικης αττικης,2011-06,0
2,αττικης,ανατολικης αττικης,2011-07,0
3,αττικης,ανατολικης αττικης,2011-08,0
4,αττικης,ανατολικης αττικης,2011-09,0
...,...,...,...,...
331,αττικης,πειραιως & νησων,2021-06,0
332,αττικης,πειραιως & νησων,2021-07,0
333,αττικης,πειραιως & νησων,2021-08,0
334,αττικης,πειραιως & νησων,2021-09,0


In [24]:
df_timeline['dt_placement']= pd.to_datetime(df_timeline['dt_placement'], format='%Y-%m').dt.to_period('M')

In [25]:
df.reset_index(inplace = True, drop=True)
df_timeline.reset_index(inplace = True, drop=True)

In [26]:
rearranged_cols = ['nuts2',	'nuts3', 'dt_placement', 'cases']

# Reindex the DataFrame with the desired column order
df = df.reindex(columns=rearranged_cols)

In [27]:
df

,nuts2,nuts3,dt_placement,cases
0,αττικης,ανατολικης αττικης,2011-07,7
1,αττικης,ανατολικης αττικης,2011-08,8
2,αττικης,βορειου τομεα αθηνων,2011-08,1
3,αττικης,δυτικης αττικης,2011-08,1
4,αττικης,πειραιως & νησων,2011-08,1
...,...,...,...,...
62,αττικης,ανατολικης αττικης,2020-07,1
63,αττικης,ανατολικης αττικης,2020-08,1
64,αττικης,ανατολικης αττικης,2021-07,1
65,αττικης,ανατολικης αττικης,2021-08,1


In [28]:
df_timeline

,nuts2,nuts3,dt_placement,cases
0,αττικης,ανατολικης αττικης,2011-05,0
1,αττικης,ανατολικης αττικης,2011-06,0
2,αττικης,ανατολικης αττικης,2011-07,0
3,αττικης,ανατολικης αττικης,2011-08,0
4,αττικης,ανατολικης αττικης,2011-09,0
...,...,...,...,...
331,αττικης,πειραιως & νησων,2021-06,0
332,αττικης,πειραιως & νησων,2021-07,0
333,αττικης,πειραιως & νησων,2021-08,0
334,αττικης,πειραιως & νησων,2021-09,0


In [29]:
merged_timeline = pd.merge(df_timeline, df, how ='left', on=['nuts2','nuts3','dt_placement'])

In [30]:
merged_timeline

,nuts2,nuts3,dt_placement,cases_x,cases_y
0,αττικης,ανατολικης αττικης,2011-05,0,NaN
1,αττικης,ανατολικης αττικης,2011-06,0,NaN
2,αττικης,ανατολικης αττικης,2011-07,0,7.0
3,αττικης,ανατολικης αττικης,2011-08,0,8.0
4,αττικης,ανατολικης αττικης,2011-09,0,5.0
...,...,...,...,...,...
331,αττικης,πειραιως & νησων,2021-06,0,NaN
332,αττικης,πειραιως & νησων,2021-07,0,NaN
333,αττικης,πειραιως & νησων,2021-08,0,NaN
334,αττικης,πειραιως & νησων,2021-09,0,NaN


In [31]:
merged_timeline['cases_y'] = merged_timeline['cases_y'].fillna(0)

In [32]:
merged_timeline['cases'] = merged_timeline['cases_x'] + merged_timeline['cases_y']
merged_timeline = merged_timeline.drop(columns=['cases_x', 'cases_y'])
merged_timeline['cases'] = merged_timeline['cases'].astype(int)

In [33]:
merged_timeline

,nuts2,nuts3,dt_placement,cases
0,αττικης,ανατολικης αττικης,2011-05,0
1,αττικης,ανατολικης αττικης,2011-06,0
2,αττικης,ανατολικης αττικης,2011-07,7
3,αττικης,ανατολικης αττικης,2011-08,8
4,αττικης,ανατολικης αττικης,2011-09,5
...,...,...,...,...
331,αττικης,πειραιως & νησων,2021-06,0
332,αττικης,πειραιως & νησων,2021-07,0
333,αττικης,πειραιως & νησων,2021-08,0
334,αττικης,πειραιως & νησων,2021-09,0


In [34]:
merged_timeline['cases'].value_counts().sort_index()

0     269
1      35
2       3
3       2
4       5
     ... 
15      1
16      1
18      1
20      1
33      1
Name: cases, Length: 18, dtype: int64

In [35]:
merged_timeline

,nuts2,nuts3,dt_placement,cases
0,αττικης,ανατολικης αττικης,2011-05,0
1,αττικης,ανατολικης αττικης,2011-06,0
2,αττικης,ανατολικης αττικης,2011-07,7
3,αττικης,ανατολικης αττικης,2011-08,8
4,αττικης,ανατολικης αττικης,2011-09,5
...,...,...,...,...
331,αττικης,πειραιως & νησων,2021-06,0
332,αττικης,πειραιως & νησων,2021-07,0
333,αττικης,πειραιως & νησων,2021-08,0
334,αττικης,πειραιως & νησων,2021-09,0


In [36]:
merged_timeline.cases.sum()

296

In [37]:
merged_timeline.to_csv(f"../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Cases_NUTS3_2010-2022.csv", encoding = enc, index = False)